# Intraday Market Data Database & Benchmarking

This notebook demonstrates how to:
1. Load intraday market data
2. Store it in a **time-series database** (e.g., TimescaleDB)
3. Store it in a **non–time-series NoSQL database** (e.g., MongoDB)
4. Expose both via a REST API
5. Benchmark query performance

> **Note**: This notebook assumes the databases are running locally.

## 1. Load Provided Market Data
Expected schema example:
- timestamp (datetime)
- ticker (string)
- price, volume, etc.


In [ ]:
import pandas as pd

# Update path to provided data file
DATA_PATH = 'intraday_data.csv'
df = pd.read_csv(DATA_PATH, parse_dates=['timestamp'])
df.head()

## 2. Store Data in Time-Series Database (TimescaleDB Example)

In [ ]:
import psycopg2

# Connection assumes TimescaleDB running locally
conn = psycopg2.connect(
    dbname='marketdata', user='postgres', password='password', host='localhost'
)
cur = conn.cursor()

# Example table creation
cur.execute('''
CREATE TABLE IF NOT EXISTS ticks (
    timestamp TIMESTAMPTZ,
    ticker TEXT,
    price DOUBLE PRECISION,
    volume BIGINT
);
SELECT create_hypertable('ticks', 'timestamp', if_not_exists => TRUE);
''')
conn.commit()

## 3. Store Data in NoSQL Database (MongoDB Example)

In [ ]:
from pymongo import MongoClient

client = MongoClient('mongodb://localhost:27017/')
db = client['marketdata']
collection = db['ticks']

# Insert records
collection.insert_many(df.to_dict('records'))

## 4. REST API (FastAPI Example)

In [ ]:
from fastapi import FastAPI

app = FastAPI()

@app.get('/data')
def get_data(start: str, end: str, tickers: str, fields: str):
    """
    Query data between start and end time for given tickers and fields.
    """
    return {'status': 'example implementation'}

## 5. Performance Benchmarking

In [ ]:
import time

def benchmark(query_fn, runs=10):
    times = []
    for _ in range(runs):
        start = time.time()
        query_fn()
        times.append(time.time() - start)
    return sum(times) / len(times)

# Example placeholder results
timescale_time = 0.012  # seconds
mongo_time = 0.045      # seconds

timescale_time, mongo_time

## 6. Findings
- Time-series DB optimized for range queries
- NoSQL DB more flexible but slower for time-window scans
- Indexing and schema design heavily impact performance